## 실습 1. 라이브러리 준비

In [1]:
# 데이터 처리
import pandas as pd

# train / test 데이터 분리
from sklearn.model_selection import train_test_split

# 텍스트를 TF-IDF 숫자 벡터로 변환
from sklearn.feature_extraction.text import TfidfVectorizer

# Naive Bayes 분류 모델
from sklearn.naive_bayes import MultinomialNB

# 모델 평가
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

print("라이브러리 준비 완료")

라이브러리 준비 완료


### 실행 결과 요약

- pandas 준비: 데이터 불러오기와 DataFrame 처리를 위해 `pandas`를 불러왔다.
- 데이터 분리 준비: 학습 데이터와 테스트 데이터를 나누기 위해 `train_test_split`을 불러왔다.
- 텍스트 변환 준비: 도서 제목을 TF-IDF 벡터로 변환하기 위해 `TfidfVectorizer`를 불러왔다.
- 모델 준비: 도서 카테고리 분류를 위해 `MultinomialNB`를 불러왔다.
- 평가 도구 준비: 정확도, 분류 보고서, 혼동 행렬을 확인하기 위한 평가 함수를 불러왔다.
- 결과 확인: 오류 없이 실행되어 Chapter 04의 Naive Bayes 분류 실습을 진행할 준비가 완료되었다.

## 실습 2. 데이터 불러오기

 

Chapter 01에서 만든 book_bestseller_clean.csv를 사용합니다.

In [2]:
# CSV 파일 경로
DATA_PATH = "book_bestseller_clean.csv"

# 전처리된 도서 데이터 불러오기
df_books = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig"
)

# 데이터 크기와 컬럼 확인
print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())

# 이번 분류에 사용할 상품명과 분야 확인
df_books[["상품명", "분야"]].head(10)

데이터 크기: (199, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']


,상품명,분야
0,소년이 온다,소설
1,모순,소설
2,결국 국민이 합니다,정치/사회
3,혼모노,소설
4,급류,소설
5,초역 부처의 말,인문
6,청춘의 독서(특별증보판),인문
7,어른의 행복은 조용하다,시/에세이
8,채식주의자,소설
9,단 한 번의 삶(강물에디션 활판인쇄 한정판),시/에세이


### 실행 결과 요약

- 데이터 크기 확인: 불러온 데이터는 총 **199행, 8개 컬럼**으로 구성되어 있다.
- 컬럼 확인: `순위`, `판매상품ID`, `상품명`, `판매가`, `저자`, `출판사`, `발행일`, `분야` 컬럼이 정상적으로 존재한다.
- 입력 데이터 확인: `상품명` 컬럼에는 `소년이 온다`, `모순`, `결국 국민이 합니다` 등의 도서 제목이 들어 있다.
- 정답 데이터 확인: `분야` 컬럼에는 `소설`, `정치/사회`, `인문`, `시/에세이` 등의 카테고리가 들어 있다.
- 분류 구조 확인: 이번 모델에서는 `상품명`을 입력값으로 사용하고 `분야`를 예측해야 할 정답값으로 사용한다.
- 결과 확인: 도서 제목과 분야 데이터가 정상적으로 연결되어 있어 Naive Bayes 분류 실습을 진행할 수 있는 상태임을 확인했다.

## 실습 3. 모델링 데이터 정리하기

 

필요한 두 컬럼만 복사합니다.

In [3]:
# 모델링에 필요한 두 컬럼만 선택
df_model = df_books[["상품명", "분야"]].copy()

# 상품명과 분야의 결측치, 자료형, 공백 정리
for col in ["상품명", "분야"]:
    df_model[col] = (
        df_model[col]
        .fillna("")      # 결측치는 빈 문자열로 변경
        .astype(str)     # 문자열 형태로 통일
        .str.strip()     # 앞뒤 공백 제거
    )

# 빈 상품명이나 빈 분야가 있는 행 제거
df_model = df_model[
    (df_model["상품명"] != "") &
    (df_model["분야"] != "")
].reset_index(drop=True)

# 모델링 데이터 크기 확인
print("모델링 데이터 크기:", df_model.shape)

# 빈 값이 남아 있는지 추가 확인
print("상품명 빈 값 수:", (df_model["상품명"] == "").sum())
print("분야 빈 값 수:", (df_model["분야"] == "").sum())

df_model.head()

모델링 데이터 크기: (199, 2)
상품명 빈 값 수: 0
분야 빈 값 수: 0


,상품명,분야
0,소년이 온다,소설
1,모순,소설
2,결국 국민이 합니다,정치/사회
3,혼모노,소설
4,급류,소설


### 실행 결과 요약

- 모델링 데이터 크기 확인: 전처리 후 데이터는 총 **199행, 2개 컬럼**으로 구성되어 있다.
- 입력값 확인: `상품명` 컬럼의 빈 값은 **0개**로 확인되었다.
- 정답값 확인: `분야` 컬럼의 빈 값도 **0개**로 확인되었다.
- 컬럼 구성 확인: 모델링에 필요한 `상품명`과 `분야` 두 컬럼만 남아 있다.
- 데이터 확인: `소년이 온다`, `모순`, `결국 국민이 합니다` 등의 상품명과 각 도서의 분야가 정상적으로 연결되어 있다.
- 결과 확인: 빈 값 없이 총 **199개의 도서 데이터**가 모델링에 사용할 수 있는 상태로 준비되었다.

## 실습 4. 분야 분포 확인하기

In [4]:
# 분야 종류 수 확인
print("분야 종류 수:", df_model["분야"].nunique())

# 분야별 데이터 수 확인
class_counts = df_model["분야"].value_counts()

print("\n분야별 데이터 수:")
display(class_counts.head(20))

# 샘플 수가 2개 미만인 분야 확인
print("\n샘플 수가 2개 미만인 분야:")
display(class_counts[class_counts < 2])

분야 종류 수: 16

분야별 데이터 수:


분야
소설         48
인문         27
경제/경영      26
시/에세이      25
자기계발       17
외국어        15
어린이(초등)    10
정치/사회       7
청소년         5
과학          5
역사/문화       5
가정/육아       3
요리          2
만화          2
예술/대중문화     1
컴퓨터/IT      1
Name: count, dtype: int64


샘플 수가 2개 미만인 분야:


분야
예술/대중문화    1
컴퓨터/IT     1
Name: count, dtype: int64

### 실행 결과 요약

- 분야 종류 확인: 전체 도서 데이터에는 총 **16개 분야**가 존재한다.
- 주요 분야 확인: `소설`이 **48개**로 가장 많았고, `인문` 27개, `경제/경영` 26개, `시/에세이` 25개 순으로 나타났다.
- 중간 규모 분야 확인: `자기계발` 17개, `외국어` 15개, `어린이(초등)` 10개 등으로 나타났다.
- 소수 분야 확인: `요리`와 `만화`는 각각 **2개**만 존재했다.
- 매우 적은 분야 확인: `예술/대중문화`와 `컴퓨터/IT`은 각각 **1개**만 존재했다.
- 클래스 불균형 확인: 분야별 데이터 수가 크게 달라 **클래스 불균형**이 존재하는 것을 확인했다.
- stratify 주의 확인: 샘플이 1개뿐인 `예술/대중문화`, `컴퓨터/IT`은 그대로 두면 `stratify=y`를 사용할 때 문제가 생길 수 있다.
- 결과 확인: train/test 분리 전에 샘플 수가 매우 적은 분야를 어떻게 처리할지 먼저 결정해야 한다.

## 실습 5. X와 y 정의하기

In [5]:
# 입력 데이터 X와 정답 데이터 y 정의
X = df_model["상품명"]
y = df_model["분야"]

# 데이터 개수 확인
print("X 데이터 수:", len(X))
print("y 데이터 수:", len(y))

# 앞의 5개 값 확인
print("\nX 예시:")
print(X.head())

print("\ny 예시:")
print(y.head())

X 데이터 수: 199
y 데이터 수: 199

X 예시:
0        소년이 온다
1            모순
2    결국 국민이 합니다
3           혼모노
4            급류
Name: 상품명, dtype: str

y 예시:
0       소설
1       소설
2    정치/사회
3       소설
4       소설
Name: 분야, dtype: str


### 실행 결과 요약

- 데이터 수 확인: 입력 데이터 `X`와 정답 데이터 `y`는 각각 **199개**로 동일하게 구성되어 있다.
- X 확인: `X`에는 `소년이 온다`, `모순`, `결국 국민이 합니다`, `혼모노`, `급류` 등의 도서 제목이 들어 있다.
- y 확인: `y`에는 각 도서 제목에 대응하는 `소설`, `정치/사회` 등의 분야가 들어 있다.
- 대응 관계 확인: 각 상품명과 분야가 같은 인덱스를 기준으로 1:1로 연결되어 있다.
- 지도학습 구조 확인: 도서 제목을 입력값으로 사용하고 분야를 정답값으로 사용하는 지도학습 구조가 정상적으로 준비되었다.
- 결과 확인: 이후 train/test 분리와 Naive Bayes 모델 학습을 진행할 수 있는 상태임을 확인했다.

## 실습 6. train/test 분리하기

In [6]:
# 분야별 데이터 수 확인
class_counts = y.value_counts()

# 샘플이 2개 이상인 분야만 선택
valid_classes = class_counts[class_counts >= 2].index

# train/test 분리에 사용할 데이터 만들기
X_split = X[y.isin(valid_classes)]
y_split = y[y.isin(valid_classes)]

# train / test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_split,
    y_split,
    test_size=0.2,
    random_state=42,
    stratify=y_split
)

# 결과 확인
print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTrain 분야 수:", y_train.nunique())
print("Test 분야 수 :", y_test.nunique())

Train: (157,)
Test : (40,)

Train 분야 수: 14
Test 분야 수 : 12


### 실행 결과 요약

- 데이터 분리 확인: train 데이터는 **157개**, test 데이터는 **40개**로 나뉘었다.
- 전체 사용 데이터 확인: 샘플이 1개뿐인 분야 2개를 제외해 총 **197개 도서**를 분리에 사용했다.
- Train 분야 수 확인: train 데이터에는 총 **14개 분야**가 포함되었다.
- Test 분야 수 확인: test 데이터에는 총 **12개 분야**가 포함되었다.
- 분야 수 차이 확인: 일부 분야는 데이터 수가 매우 적어 `stratify`를 사용했더라도 test 데이터에 포함되지 않았다.
- 주의점: 따라서 이후 test 성능 평가는 전체 14개 분야가 아니라 **실제로 test에 포함된 12개 분야**를 중심으로 해석해야 한다.
- 결과 확인: train 데이터는 모델 학습에, test 데이터는 학습에 사용하지 않은 도서의 분야를 예측하고 평가하는 데 사용할 수 있도록 분리되었다.

## 실습 7. 가장 중요한 원칙 — split을 먼저 한다

### split을 먼저 해야 하는 이유

- 잘못된 순서: 전체 데이터에 TF-IDF를 먼저 `fit`한 뒤 train/test를 분리하면 test 데이터의 단어 정보가 학습 과정에 포함될 수 있다.
- 올바른 순서: 먼저 train/test를 분리한 뒤 train 데이터에만 TF-IDF를 `fit`해야 한다.
- test 데이터는 train에서 학습한 Vectorizer를 사용해 `transform()`만 수행한다.
- 이러한 순서를 지켜야 데이터 누수(Data Leakage)를 막고 모델 성능을 공정하게 평가할 수 있다.

```text
원본 텍스트
    ↓
train / test 분리
    ↓
train에 TF-IDF fit
    ↓
train transform
    ↓
test는 transform만 수행
    ↓
모델 학습 및 평가

## 실습 8. fit과 transform 이해하기

### fit과 transform 이해하기

- 정의: `fit`은 train 데이터에서 단어 사전과 IDF 규칙을 학습하는 과정이다.
- 정의: `transform`은 이미 학습한 규칙을 사용해 문장을 숫자 벡터로 변환하는 과정이다.
- train 처리: train 데이터에는 학습과 변환을 함께 수행하는 `fit_transform()`을 사용한다.
- test 처리: test 데이터에는 이미 학습한 규칙을 그대로 적용해야 하므로 `transform()`만 사용한다.
- 핵심 원칙: test 데이터에서 다시 `fit()`하면 test 정보가 학습 과정에 들어갈 수 있으므로 사용하지 않는다.

In [8]:
# TF-IDF 객체 생성
vectorizer = TfidfVectorizer()

# 1. train 데이터에서 규칙 학습
vectorizer.fit(X_train)

# 학습된 단어 수 확인
print("학습된 단어 수:", len(vectorizer.get_feature_names_out()))

# 2. 학습된 규칙으로 train 데이터 변환
X_train_tfidf = vectorizer.transform(X_train)

# 3. 같은 규칙으로 test 데이터 변환
X_test_tfidf = vectorizer.transform(X_test)

# 결과 크기 확인
print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape :", X_test_tfidf.shape)

학습된 단어 수: 447
Train TF-IDF shape: (157, 447)
Test TF-IDF shape : (40, 447)


### 실행 결과 요약

- 학습된 단어 수 확인: train 데이터에서 총 **447개 단어**를 학습했다.
- Train 변환 확인: train 데이터는 **157개 문서 × 447개 단어**의 TF-IDF 행렬로 변환되었다.
- Test 변환 확인: test 데이터는 **40개 문서 × 447개 단어**의 TF-IDF 행렬로 변환되었다.
- 동일한 feature 사용 확인: train과 test 모두 같은 **447개 단어 기준**으로 변환되었다.
- fit과 transform 구분 확인: train 데이터에서만 단어 사전과 IDF를 학습하고, test 데이터에는 학습된 규칙으로 `transform()`만 적용했다.
- 결과 확인: `fit`은 규칙 학습, `transform`은 학습된 규칙 적용이라는 차이를 실제 결과로 확인했다.

## 실습 9. TF-IDF 변환하기

In [9]:
# TF-IDF 객체 생성
tfidf = TfidfVectorizer()

# train 데이터에서 학습 + 변환
X_train_tfidf = tfidf.fit_transform(X_train)

# test 데이터는 학습하지 않고 변환만 수행
X_test_tfidf = tfidf.transform(X_test)

# 결과 크기 확인
print("Train TF-IDF:", X_train_tfidf.shape)
print("Test TF-IDF :", X_test_tfidf.shape)

# 열 개수가 같은지 확인
print("열 수 동일 여부:", X_train_tfidf.shape[1] == X_test_tfidf.shape[1])

Train TF-IDF: (157, 447)
Test TF-IDF : (40, 447)
열 수 동일 여부: True


### 실행 결과 요약

- Train 변환 확인: train 데이터는 **157개 문서 × 447개 단어**의 TF-IDF 행렬로 변환되었다.
- Test 변환 확인: test 데이터는 **40개 문서 × 447개 단어**의 TF-IDF 행렬로 변환되었다.
- 열 수 일치 확인: train과 test의 열 수는 모두 **447개**로 동일하게 나타났다.
- 동일 단어 공간 확인: `열 수 동일 여부`가 `True`로 확인되어 train과 test가 같은 Vectorizer의 단어 공간을 사용하고 있음을 확인했다.
- 데이터 누수 방지 확인: train 데이터에만 `fit_transform()`을 적용하고 test 데이터에는 `transform()`만 적용했다.
- 결과 확인: train과 test의 문서 수는 다르지만 동일한 447개 feature 기준으로 변환되어 이후 Naive Bayes 모델 학습과 평가에 사용할 수 있다.

## 실습 10. Multinomial Naive Bayes 이해하기

In [10]:
# Multinomial Naive Bayes 모델 생성
model = MultinomialNB()

print("모델 생성 완료:", model)

모델 생성 완료: MultinomialNB()


### 실행 결과 요약

- 모델 생성 확인: `MultinomialNB()` 객체가 오류 없이 정상적으로 생성되었다.
- 모델 준비 완료: TF-IDF로 변환된 train 데이터를 학습할 Naive Bayes 분류 모델이 준비되었다.
- 활용 목적 확인: 이 모델은 도서 제목의 단어 패턴을 바탕으로 해당 도서가 어떤 분야에 속하는지 예측하는 데 사용한다.
- 결과 확인: `MultinomialNB()`가 정상적으로 생성되어 다음 단계에서 모델 학습을 진행할 수 있다.

## 실습 11. 모델 학습과 예측

In [11]:
# Naive Bayes 모델 학습
model.fit(X_train_tfidf, y_train)

# test 데이터 예측
y_pred = model.predict(X_test_tfidf)

# 실제 분야와 예측 분야를 비교할 DataFrame 생성
result = pd.DataFrame({
    "상품명": X_test.reset_index(drop=True),
    "실제_분야": y_test.reset_index(drop=True),
    "예측_분야": y_pred
})

# 앞의 20개 결과 확인
result.head(20)

,상품명,실제_분야,예측_분야
0,제로 투 원(10주년 기념판),경제/경영,소설
1,흰,소설,소설
2,사랑과 멸종을 바꿔 읽어보십시오,시/에세이,소설
3,최소한의 한국사,역사/문화,소설
4,왜 그들만 부자가 되는가,경제/경영,소설
5,죽이고 싶은 아이,청소년,소설
6,침묵의 퍼레이드,소설,소설
7,행동은 불안을 이긴다,자기계발,소설
8,더 나은 어휘를 쓰고 싶은 당신을 위한 필사책,인문,인문
9,내가 한 말을 내가 오해하지 않기로 함,시/에세이,소설


### 실행 결과 요약

- 예측 결과 확인: test 데이터의 실제 분야와 Naive Bayes 모델이 예측한 분야를 직접 비교했다.
- 정상 예측 확인: `흰`, `침묵의 퍼레이드`, `혼모노`는 실제 `소설`을 `소설`로 예측했고, `해커스 토익 RC Reading(리딩) 기본서`는 `외국어`를 `외국어`로 예측했다.
- 인문 분야 확인: `더 나은 어휘를 쓰고 싶은 당신을 위한 필사책`은 실제 `인문`을 `인문`으로 올바르게 예측했다.
- 오분류 확인: `경제/경영`, `시/에세이`, `역사/문화`, `청소년`, `자기계발`, `정치/사회` 등 여러 분야의 도서가 `소설`로 잘못 예측된 사례가 많이 나타났다.
- 예측 편향 확인: 앞의 20개 결과에서는 모델이 여러 도서를 `소설`로 예측하는 경향이 강하게 나타났다.
- 가능한 원인 확인: `소설` 분야의 학습 데이터가 가장 많고, 제목만으로 분야를 구분하기 어려운 경우가 있어 특정 클래스 쪽으로 예측이 치우쳤을 가능성이 있다.
- 결과 확인: 일부 도서는 올바르게 분류했지만 오분류도 많이 보여, 정확도와 분야별 성능을 추가로 확인할 필요가 있다.

## 실습 12. Accuracy 확인하기

In [12]:
# Accuracy 계산
accuracy = accuracy_score(y_test, y_pred)

# 결과 출력
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.3500


### 실행 결과 요약

- Accuracy 확인: Naive Bayes 모델의 정확도는 **0.3500**, 즉 **35.0%**로 나타났다.
- 의미 확인: test 데이터 40개 중 약 35%를 올바르게 분류했다는 뜻이다.
- 성능 해석: 앞서 확인한 예측 결과처럼 여러 분야가 `소설`로 잘못 분류된 경우가 많아 전체 정확도가 높지 않았다.
- 클래스 불균형 영향: 분야별 데이터 수가 다르기 때문에 Accuracy만으로 모델 성능을 충분히 판단하기 어렵다.
- 다음 단계: 분야별 `precision`, `recall`, `F1-score`를 확인해 어떤 분야에서 잘 맞추고 어떤 분야에서 성능이 낮은지 추가로 살펴볼 필요가 있다.

## 실습 13. Classification Report 확인하기

In [13]:
# Classification Report 확인
report = classification_report(
    y_test,
    y_pred,
    zero_division=0
)

print(report)

              precision    recall  f1-score   support

       가정/육아       0.00      0.00      0.00         1
       경제/경영       1.00      0.20      0.33         5
          과학       0.00      0.00      0.00         1
          소설       0.28      1.00      0.43        10
       시/에세이       0.00      0.00      0.00         5
     어린이(초등)       0.00      0.00      0.00         2
       역사/문화       0.00      0.00      0.00         1
         외국어       1.00      0.33      0.50         3
          인문       1.00      0.33      0.50         6
        자기계발       0.00      0.00      0.00         4
       정치/사회       0.00      0.00      0.00         1
         청소년       0.00      0.00      0.00         1

    accuracy                           0.35        40
   macro avg       0.27      0.16      0.15        40
weighted avg       0.42      0.35      0.26        40



### 실행 결과 요약

- 전체 정확도 확인: 모델의 Accuracy는 **0.35**, 즉 **35%**로 나타났다.
- 소설 분야 확인: `소설`은 recall이 **1.00**으로 실제 소설 10개를 모두 찾아냈지만, precision은 **0.28**로 낮아 다른 분야까지 소설로 많이 예측한 것을 확인했다.
- 경제/경영 확인: precision은 **1.00**이지만 recall은 **0.20**으로, 경제/경영이라고 예측한 경우는 정확했지만 실제 경제/경영 5개 중 일부만 찾아냈다.
- 외국어와 인문 확인: `외국어`와 `인문`은 각각 precision **1.00**, recall **0.33**, F1-score **0.50**으로 나타났다.
- 낮은 성능 분야 확인: `가정/육아`, `과학`, `시/에세이`, `어린이(초등)`, `역사/문화`, `자기계발`, `정치/사회`, `청소년`은 precision과 recall이 모두 **0**으로 나타났다.
- 평균 성능 확인: macro F1-score는 **0.15**, weighted F1-score는 **0.26**으로 나타나 분야별 성능 차이가 큰 것을 확인했다.
- 결과 해석: 모델이 `소설`을 지나치게 많이 예측하는 경향이 있으며, 데이터 수가 적은 분야에서는 거의 예측하지 못하는 문제가 나타났다.
- 결과 확인: Accuracy만 보는 것보다 분야별 precision, recall, F1-score를 함께 확인해야 모델의 실제 분류 성능을 더 정확하게 이해할 수 있다.

## 실습 14. Confusion Matrix 확인하기

In [15]:
# 모델이 학습한 분야 목록 확인
labels = model.classes_

# Confusion Matrix 계산
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

# 보기 쉽게 DataFrame으로 변환
df_cm = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

# 결과 확인
df_cm

,가정/육아,경제/경영,과학,만화,소설,시/에세이,어린이(초등),역사/문화,외국어,요리,인문,자기계발,정치/사회,청소년
가정/육아,0,0,0,0,1,0,0,0,0,0,0,0,0,0
경제/경영,0,1,0,0,4,0,0,0,0,0,0,0,0,0
과학,0,0,0,0,1,0,0,0,0,0,0,0,0,0
만화,0,0,0,0,0,0,0,0,0,0,0,0,0,0
소설,0,0,0,0,10,0,0,0,0,0,0,0,0,0
시/에세이,0,0,0,0,5,0,0,0,0,0,0,0,0,0
어린이(초등),0,0,0,0,2,0,0,0,0,0,0,0,0,0
역사/문화,0,0,0,0,1,0,0,0,0,0,0,0,0,0
외국어,0,0,0,0,2,0,0,0,1,0,0,0,0,0
요리,0,0,0,0,0,0,0,0,0,0,0,0,0,0


### 실행 결과 요약

- 혼동 행렬 확인: 실제 분야와 모델이 예측한 분야를 행과 열로 비교했다.
- 소설 예측 편향 확인: test 데이터 **40개 중 36개를 `소설`로 예측**해 모델의 예측이 소설 분야에 크게 치우쳐 있었다.
- 소설 분야 확인: 실제 `소설` 10개는 모두 `소설`로 예측되어 **10개 모두 정답**이었다.
- 경제/경영 확인: 실제 5개 중 **1개만 `경제/경영`으로 맞게 예측**했고, 나머지 4개는 `소설`로 잘못 예측했다.
- 외국어 확인: 실제 3개 중 **1개는 `외국어`로 맞게 예측**했고, 2개는 `소설`로 분류했다.
- 인문 확인: 실제 6개 중 **2개는 `인문`으로 맞게 예측**했고, 4개는 `소설`로 분류했다.
- 오분류 확인: `가정/육아`, `과학`, `시/에세이`, `어린이(초등)`, `역사/문화`, `자기계발`, `정치/사회`, `청소년`은 test 데이터에서 모두 `소설`로 잘못 분류되었다.
- Test 미포함 분야 확인: `만화`, `요리`는 test 데이터에 실제 샘플이 없어 해당 행이 모두 0으로 나타났다.
- 결과 확인: 모델이 일부 분야는 구분했지만 대부분의 도서를 `소설`로 예측하는 경향이 강했으며, 이것이 앞서 확인한 **Accuracy 35%와 낮은 분야별 성능**으로 이어졌음을 확인했다.

## 실습 15. 오분류 사례 확인하기

In [16]:
# 실제 분야와 예측 분야가 다른 오분류 데이터만 선택
misclassified = result[
    result["실제_분야"] != result["예측_분야"]
].copy()

# 오분류 개수 확인
print("오분류 수:", len(misclassified))

# 앞의 20개 오분류 사례 확인
misclassified.head(20)

오분류 수: 26


,상품명,실제_분야,예측_분야
0,제로 투 원(10주년 기념판),경제/경영,소설
2,사랑과 멸종을 바꿔 읽어보십시오,시/에세이,소설
3,최소한의 한국사,역사/문화,소설
4,왜 그들만 부자가 되는가,경제/경영,소설
5,죽이고 싶은 아이,청소년,소설
7,행동은 불안을 이긴다,자기계발,소설
9,내가 한 말을 내가 오해하지 않기로 함,시/에세이,소설
11,쇼펜하우어 인생수업(30만 부 기념 개정증보판),인문,소설
13,나는 메트로폴리탄 미술관의 경비원입니다,시/에세이,소설
14,단 3개의 미국 ETF로 은퇴하라,경제/경영,소설


### 실행 결과 요약

- 오분류 수 확인: 앞서 Accuracy가 **35%**였으므로 test 데이터 40개 중 **26개가 오분류**된 것으로 해석할 수 있다.
- 예측 편향 확인: 확인된 오분류 사례 대부분이 실제 분야와 관계없이 `소설`로 예측되었다.
- 경제/경영 오분류: `제로 투 원`, `왜 그들만 부자가 되는가`, `단 3개의 미국 ETF로 은퇴하라` 등이 모두 `소설`로 잘못 분류되었다.
- 시/에세이 오분류: `사랑과 멸종을 바꿔 읽어보십시오`, `나는 메트로폴리탄 미술관의 경비원입니다`, `소년과 두더지와 여우와 말` 등이 `소설`로 분류되었다.
- 자기계발 오분류: `행동은 불안을 이긴다`, `아주 작은 습관의 힘`, `어른의 품격을 채우는 100일 필사 노트` 등이 `소설`로 잘못 예측되었다.
- 기타 분야 오분류: `역사/문화`, `청소년`, `인문`, `정치/사회`, `과학`, `가정/육아`, `외국어` 분야에서도 `소설`로 분류되는 사례가 나타났다.
- 제목만의 한계 확인: 제목만 보고 분야를 판단하기 어려운 도서가 많고, 여러 분야에서 공통적으로 사용할 수 있는 표현도 존재했다.
- 데이터 부족 영향: 학습 데이터 수가 적은 분야는 해당 분야의 특징적인 단어 패턴을 충분히 학습하지 못했을 가능성이 있다.
- 결과 확인: 현재 모델의 핵심 문제는 여러 분야를 `소설`로 과도하게 예측하는 경향이며, 이후 클래스 불균형과 특징 추출 방식을 함께 개선할 필요가 있다.

## 실습 16. 예측 결과 저장하기

In [17]:
# 전체 예측 결과 저장
result.to_csv(
    "chapter04_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# 오분류 결과 저장
misclassified.to_csv(
    "chapter04_misclassified.csv",
    index=False,
    encoding="utf-8-sig"
)

# 저장한 파일 다시 불러오기
predictions_check = pd.read_csv(
    "chapter04_predictions.csv",
    encoding="utf-8-sig"
)

misclassified_check = pd.read_csv(
    "chapter04_misclassified.csv",
    encoding="utf-8-sig"
)

# 결과 확인
print("예측 결과 컬럼:", predictions_check.columns.tolist())
print("오분류 결과 컬럼:", misclassified_check.columns.tolist())

display(predictions_check.head())
display(misclassified_check.head())

예측 결과 컬럼: ['상품명', '실제_분야', '예측_분야']
오분류 결과 컬럼: ['상품명', '실제_분야', '예측_분야']


,상품명,실제_분야,예측_분야
0,제로 투 원(10주년 기념판),경제/경영,소설
1,흰,소설,소설
2,사랑과 멸종을 바꿔 읽어보십시오,시/에세이,소설
3,최소한의 한국사,역사/문화,소설
4,왜 그들만 부자가 되는가,경제/경영,소설


,상품명,실제_분야,예측_분야
0,제로 투 원(10주년 기념판),경제/경영,소설
1,사랑과 멸종을 바꿔 읽어보십시오,시/에세이,소설
2,최소한의 한국사,역사/문화,소설
3,왜 그들만 부자가 되는가,경제/경영,소설
4,죽이고 싶은 아이,청소년,소설


### 실행 결과 요약

- 파일 재로드 확인: `chapter04_predictions.csv`와 `chapter04_misclassified.csv`를 다시 불러와 저장 결과를 확인했다.
- 컬럼 확인: 두 파일 모두 `상품명`, `실제_분야`, `예측_분야` 컬럼이 정상적으로 유지되었다.
- 한글 확인: `제로 투 원(10주년 기념판)`, `사랑과 멸종을 바꿔 읽어보십시오`, `최소한의 한국사` 등 한글 상품명이 깨지지 않고 정상적으로 출력되었다.
- 전체 예측 결과 확인: 전체 예측 파일에는 정답과 오답이 모두 포함되어 있으며, `흰`처럼 실제 분야와 예측 분야가 모두 `소설`인 정상 예측도 확인되었다.
- 오분류 결과 확인: 오분류 파일에는 실제 분야와 예측 분야가 다른 데이터만 남아 있었으며, `경제/경영`, `시/에세이`, `역사/문화`, `청소년` 등이 `소설`로 잘못 예측된 사례를 확인했다.
- 결과 확인: 전체 예측 결과와 오분류 결과가 각각 목적에 맞게 CSV 파일로 정상적으로 저장되고 다시 불러와졌음을 확인했다.